# VisDrone2019 DET: YOLO11s augmentation baseline

Run cells from top to bottom in Kaggle. This notebook converts VisDrone CSV annotations to YOLO labels, measures image/box conditions, creates targeted **training-only** photometric/degradation/occlusion variants, and trains a stock YOLO11s baseline at 1280px. Validation images and labels remain original.

Geometric transforms (flip, scale/translation, and mosaic) are applied by Ultralytics during training so their boxes stay aligned. This is a baseline; custom P2 heads/downsampling/attention/loss variants should be evaluated as separate experiments.


In [ ]:
!pip install -q ultralytics

from pathlib import Path
from ultralytics.utils.downloads import download

DATA_ROOT = Path('/kaggle/working/VisDrone')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
URLS = [
    'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip',
    'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip',
]
download(URLS, dir=DATA_ROOT, threads=2)

for split_name in ('VisDrone2019-DET-train', 'VisDrone2019-DET-val'):
    split_dir = DATA_ROOT / split_name
    if not (split_dir / 'images').is_dir() or not (split_dir / 'annotations').is_dir():
        raise FileNotFoundError(
            f'Expected images/ and annotations/ under {split_dir}. '
            'Check that both archives downloaded and extracted successfully.'
        )
print('Train and validation archives are ready.')


In [ ]:
import cv2
import numpy as np
import pandas as pd

SPLIT_DIRS = {
    'train': DATA_ROOT / 'VisDrone2019-DET-train',
    'val': DATA_ROOT / 'VisDrone2019-DET-val',
}
BOX_COLUMNS = ['x', 'y', 'w', 'h', 'score', 'category', 'truncation', 'occlusion']
CLASS_NAMES = [
    'pedestrian', 'people', 'bicycle', 'car', 'van', 'truck',
    'tricycle', 'awning-tricycle', 'bus', 'motor',
]

image_rows = []
box_rows = []
for split, split_dir in SPLIT_DIRS.items():
    image_dir = split_dir / 'images'
    annotation_dir = split_dir / 'annotations'
    for image_path in sorted(image_dir.glob('*.jpg')):
        image = cv2.imread(str(image_path))
        if image is None:
            raise ValueError(f'Could not read image: {image_path}')
        height, width = image.shape[:2]
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        ann_path = annotation_dir / f'{image_path.stem}.txt'
        image_rows.append({
            'split': split, 'name': image_path.stem,
            'image_path': str(image_path), 'ann_path': str(ann_path),
            'img_w': width, 'img_h': height, 'img_area': width * height,
            'brightness': float(gray.mean()),
            'blur': float(cv2.Laplacian(gray, cv2.CV_64F).var()),
        })
        if not ann_path.exists():
            continue
        for object_id, line in enumerate(ann_path.read_text(encoding='utf-8').splitlines()):
            values = line.strip().rstrip(',').split(',')
            if len(values) < 8:
                continue
            box = dict(zip(BOX_COLUMNS, map(int, values[:8])))
            x, y, bw, bh = (box[k] for k in ('x', 'y', 'w', 'h'))
            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(width, x + bw), min(height, y + bh)
            if x2 <= x1 or y2 <= y1:
                continue
            crop = image[y1:y2, x1:x2]
            crop_gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
            box_rows.append({
                'split': split, 'name': image_path.stem, 'object_id': object_id,
                'image_path': str(image_path), **box,
                'category_name': CLASS_NAMES[box['category'] - 1]
                    if 1 <= box['category'] <= 10 else 'ignored_or_unknown',
                'box_area_px': (x2 - x1) * (y2 - y1),
                'box_brightness': float(crop_gray.mean()),
                'box_blur': float(cv2.Laplacian(crop_gray, cv2.CV_64F).var()),
            })

df = pd.DataFrame(image_rows)
boxes_df = pd.DataFrame(box_rows)
valid_boxes = boxes_df[
    boxes_df['score'].gt(0) & boxes_df['category'].between(1, 10)
].copy()
box_dims = valid_boxes.merge(
    df[['split', 'name', 'img_w', 'img_h']], on=['split', 'name'], validate='many_to_one')
box_scale = 1280 / box_dims[['img_w', 'img_h']].max(axis=1)
valid_boxes['box_area_at_1280'] = box_dims['box_area_px'].to_numpy() * box_scale.to_numpy() ** 2
valid_boxes['object_size'] = np.select(
    [valid_boxes['box_area_at_1280'] < 8**2,
     valid_boxes['box_area_at_1280'] < 32**2,
     valid_boxes['box_area_at_1280'] < 96**2],
    ['tiny', 'small', 'medium'], default='large')

summary = valid_boxes.groupby(['split', 'name'], as_index=False).agg(
    n_objects=('category', 'size'),
    occ_frac=('occlusion', lambda s: s.gt(0).mean()),
    trunc_frac=('truncation', lambda s: s.gt(0).mean()),
    small_ratio=('object_size', lambda s: s.isin(['tiny', 'small']).mean()),
)
df = df.merge(summary, on=['split', 'name'], how='left')
for column in ['n_objects', 'occ_frac', 'trunc_frac', 'small_ratio']:
    df[column] = df[column].fillna(0)
df['n_objects'] = df['n_objects'].astype(int)

train_mask = df['split'].eq('train')
brightness_q20, brightness_q80 = df.loc[train_mask, 'brightness'].quantile([.20, .80])
blur_q20 = df.loc[train_mask, 'blur'].quantile(.20)
df['brightness_lbl'] = pd.cut(
    df['brightness'], [-np.inf, brightness_q20, brightness_q80, np.inf],
    labels=['dark', 'normal', 'bright'])
df['blur_lbl'] = pd.cut(
    df['blur'], [-np.inf, blur_q20, np.inf], labels=['blurry', 'sharp'])
density_q80 = df.loc[train_mask, 'n_objects'].quantile(.80)
df['density_lbl'] = np.where(df['n_objects'] >= density_q80, 'crowded', 'not_crowded')

df.to_csv('/kaggle/working/all_images_labeled.csv', index=False)
boxes_df.to_csv('/kaggle/working/boxes_df.csv', index=False)
print(f'Images: {len(df):,}; parsed boxes: {len(boxes_df):,}; valid detection boxes: {len(valid_boxes):,}')
print(f'Train-derived brightness q20/q80: {brightness_q20:.1f}/{brightness_q80:.1f}; blur q20: {blur_q20:.1f}')
df.groupby(['split', 'brightness_lbl'], observed=True).size().unstack(fill_value=0)


In [ ]:
import random
import shutil

OUTPUT_ROOT = Path('/kaggle/working/VisDrone_yolo')
random.seed(42)
rng = np.random.default_rng(42)

def to_yolo_label_file(ann_path, width, height, output_path):
    lines = []
    if ann_path.exists():
        for raw_line in ann_path.read_text(encoding='utf-8').splitlines():
            values = raw_line.strip().rstrip(',').split(',')
            if len(values) < 8:
                continue
            x, y, bw, bh, score, category, _, _ = map(int, values[:8])
            if score <= 0 or not 1 <= category <= 10 or bw <= 0 or bh <= 0:
                continue
            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(width, x + bw), min(height, y + bh)
            if x2 <= x1 or y2 <= y1:
                continue
            xc, yc = (x1 + x2) / (2 * width), (y1 + y2) / (2 * height)
            nw, nh = (x2 - x1) / width, (y2 - y1) / height
            lines.append(f'{category - 1} {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}')
    output_path.write_text('\n'.join(lines), encoding='utf-8')

# Derive box-condition thresholds from valid TRAIN objects only.
train_boxes = valid_boxes[valid_boxes['split'].eq('train')]
box_dark_limit = float(train_boxes['box_brightness'].quantile(.20))
box_blur_limit = float(train_boxes['box_blur'].quantile(.20))
train_density_limit = float(df.loc[train_mask, 'n_objects'].quantile(.80))

for split, split_dir in SPLIT_DIRS.items():
    image_out = OUTPUT_ROOT / 'images' / split
    label_out = OUTPUT_ROOT / 'labels' / split
    image_out.mkdir(parents=True, exist_ok=True)
    label_out.mkdir(parents=True, exist_ok=True)
    for row in df[df['split'].eq(split)].itertuples(index=False):
        source = Path(row.image_path)
        target = image_out / source.name
        shutil.copy2(source, target)
        to_yolo_label_file(Path(row.ann_path), row.img_w, row.img_h,
                           label_out / f'{source.stem}.txt')

# Create at most one targeted photometric variant per eligible TRAIN image.
# These transforms do not move boxes; original YOLO labels are copied unchanged.
generated = []
for row in df[df['split'].eq('train')].itertuples(index=False):
    image_boxes = train_boxes[train_boxes['name'].eq(row.name)]
    if image_boxes.empty:
        continue
    dark_ratio = float(image_boxes['box_brightness'].le(box_dark_limit).mean())
    blurry_ratio = float(image_boxes['box_blur'].le(box_blur_limit).mean())
    policies = []
    image = cv2.imread(row.image_path)
    if image is None:
        raise ValueError(f'Could not read image: {row.image_path}')

    if dark_ratio >= .20:
        gamma = float(rng.uniform(.72, .92))
        lut = np.array([min(255, round(((i / 255) ** gamma) * 255))
                        for i in range(256)], dtype=np.uint8)
        image = cv2.LUT(image, lut)
        alpha = float(rng.uniform(.95, 1.10))
        beta = int(rng.integers(0, 16))
        image = cv2.convertScaleAbs(image, alpha=alpha, beta=beta)
        policies.append('low_light')

    # Degrade mostly sharp images to model distant/low-quality capture, not already-blurry examples.
    if blurry_ratio <= .20 and row.small_ratio >= .20:
        h, w = image.shape[:2]
        factor = float(rng.uniform(.65, .85))
        reduced = cv2.resize(image, (max(1, int(w * factor)), max(1, int(h * factor))),
                             interpolation=cv2.INTER_AREA)
        image = cv2.resize(reduced, (w, h), interpolation=cv2.INTER_LINEAR)
        if rng.random() < .5:
            image = cv2.GaussianBlur(image, (3, 3), sigmaX=float(rng.uniform(.2, .8)))
        if rng.random() < .5:
            noise = rng.normal(0, rng.uniform(2, 6), image.shape).astype(np.float32)
            image = np.clip(image.astype(np.float32) + noise, 0, 255).astype(np.uint8)
        quality = int(rng.integers(55, 86))
        ok, encoded = cv2.imencode('.jpg', image, [cv2.IMWRITE_JPEG_QUALITY, quality])
        if not ok:
            raise RuntimeError(f'JPEG encoding failed for {row.name}')
        image = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
        policies.append('distance_quality')

    # Small coarse-dropout patches are a mild synthetic occlusion for crowded scenes.
    if row.n_objects >= train_density_limit and row.n_objects > 0:
        h, w = image.shape[:2]
        for _ in range(2):
            patch_w = max(1, int(w * rng.uniform(.02, .05)))
            patch_h = max(1, int(h * rng.uniform(.02, .05)))
            x0 = int(rng.integers(0, max(1, w - patch_w + 1)))
            y0 = int(rng.integers(0, max(1, h - patch_h + 1)))
            fill = image.mean(axis=(0, 1)).astype(np.uint8)
            image[y0:y0 + patch_h, x0:x0 + patch_w] = fill
        policies.append('mild_occlusion')

    if not policies:
        continue
    stem = f'{row.name}_aug'
    out_image = OUTPUT_ROOT / 'images' / 'train' / f'{stem}.jpg'
    if not cv2.imwrite(str(out_image), image):
        raise RuntimeError(f'Could not write augmented image: {out_image}')
    source_label = OUTPUT_ROOT / 'labels' / 'train' / f'{row.name}.txt'
    shutil.copy2(source_label, OUTPUT_ROOT / 'labels' / 'train' / f'{stem}.txt')
    generated.append({'name': row.name, 'augmented_name': stem, 'policies': '+'.join(policies)})

pd.DataFrame(generated).to_csv(OUTPUT_ROOT / 'targeted_augmentation_manifest.csv', index=False)

yaml_text = (
    f'path: {OUTPUT_ROOT}\n'
    'train: images/train\n'
    'val: images/val\n'
    'names: [' + ', '.join(CLASS_NAMES) + ']\n'
)
DATA_YAML = OUTPUT_ROOT / 'dataset.yaml'
DATA_YAML.write_text(yaml_text, encoding='utf-8')
print(f'YOLO dataset ready: {OUTPUT_ROOT}')
print(f'Train-only targeted variants: {len(generated):,}')
print(f'Box darkness threshold: {box_dark_limit:.1f}; box blur threshold: {box_blur_limit:.1f}')
print('Validation is original and unaugmented.')


In [ ]:
import torch
from ultralytics import YOLO

MODEL_WEIGHTS = 'yolo11s.pt'
model = YOLO(MODEL_WEIGHTS)
results = model.train(
    data=str(DATA_YAML),
    imgsz=1280,
    epochs=100,
    batch=4,
    workers=2,
    device=0 if torch.cuda.is_available() else 'cpu',
    seed=42,
    deterministic=True,
    patience=20,
    project='/kaggle/working/runs/visdrone',
    name='yolo11s_aug_1280',
    exist_ok=True,
    # Geometric augmentation is handled by Ultralytics with labels transformed alongside images.
    fliplr=0.5,
    flipud=0.0,
    degrees=0.0,
    translate=0.05,
    scale=0.30,
    shear=0.0,
    perspective=0.0,
    mosaic=0.5,
    close_mosaic=10,
    mixup=0.05,
    hsv_h=0.015,
    hsv_s=0.50,
    hsv_v=0.30,
)
print('Best checkpoint:', model.trainer.best)


## Notes

- The numeric brightness/blur cutoffs are empirical 20th/80th percentiles from the training split, not universal constants. The notebook saves per-image and per-box tables for inspection.
- Geometric augmentations and mosaics happen on training batches only; no validation annotations are transformed.
- If 1280px with batch 4 exceeds Kaggle GPU memory, lower `batch` to 2 or 1.
- This run uses the standard YOLO11s architecture and loss as a baseline. Compare P2, downsampling, attention, and alternate loss changes one at a time in separate runs.
